# OpenPlaque — low-RAM left-main acceptance gate

This run deliberately stops at the **left main**. It reuses the validated downsampled source-evidence cache and RCA calibration, keeps the evidence as memory-mapped shards, performs only short local graph searches, then releases that evidence before streaming source CCTA series 7 slice-by-slice into an `int16` memory map for orthogonal lumen QC.

The key RAM fix is that orthogonal sampling never converts the full CCTA volume to `float64`.


## Step 1 — Mount Google Drive


In [ ]:
# FIRST EXECUTABLE CELL
from google.colab import drive
drive.mount('/content/drive')


## Step 2 — Cache reuse controls


In [ ]:
REUSE_OSTIA = True
REUSE_GRAPH_CANDIDATES = True
REUSE_SOURCE_CT = True
REUSE_CANDIDATE_QC = True
REUSE_FIGURES = True
REUSE_REPORT = True


## Step 3 — Install this branch


In [ ]:
!rm -rf /content/OpenPlaque
!git clone -q --branch left-main-bifurcation-from-main --single-branch https://github.com/pazzani/OpenPlaque.git /content/OpenPlaque
%pip -q install pydicom SimpleITK scipy scikit-image matplotlib pandas psutil
import sys, gc, os, psutil
sys.path.insert(0, '/content/OpenPlaque/src')
def ram(label):
    p=psutil.Process(os.getpid())
    print(f'{label}: RSS {p.memory_info().rss/1024**3:.2f} GB')
ram('After install')


## Step 4 — Initialize the low-RAM workflow


In [ ]:
from openplaque.left_main_lowram import LeftMainLowRamWorkflow, ALGORITHM_VERSION
reuse = {
    'ostia': REUSE_OSTIA,
    'graph_candidates': REUSE_GRAPH_CANDIDATES,
    'source_ct': REUSE_SOURCE_CT,
    'candidate_qc': REUSE_CANDIDATE_QC,
    'figures': REUSE_FIGURES,
    'report': REUSE_REPORT,
}
wf = LeftMainLowRamWorkflow('/content/drive/MyDrive/OpenPlaque', reuse=reuse)
print('Algorithm:', ALGORITHM_VERSION)
print('Uses existing validated source evidence:', wf.cache/'source_evidence.npz')
print('Uses existing RCA calibration:', wf.cache/'rca_calibration.json')


## Step 5 — Stage evidence one array at a time and find ostium candidates
The compressed evidence cache is unpacked into local `.npy` shards one member at a time. The shards are memory-mapped rather than held as one large in-memory dictionary.


In [ ]:
wf.stage_evidence()
wf.load_rca_calibration()
ostia = wf.detect_ostia()
ram('After evidence + ostia')
display(ostia.head(12))


## Step 6 — Short left-main graph candidates only
Only 6–30 mm candidates are traced. Each endpoint is handled in a small local graph box. No distal LAD/LCX search is attempted in this run.


In [ ]:
graph_table = wf.build_graph_candidates()
display(graph_table.head(20))
wf.release_evidence()
gc.collect(); ram('After releasing evidence')


## Step 7 — Stream source CCTA to an int16 memory map and serially QC the candidates
Series 7 is read slice-by-slice. Orthogonal planes are sampled directly from the memory map; the full CCTA is never promoted to `float64`.


In [ ]:
wf.load_source_ct()
summary = wf.qc_candidates()
print('Best left-main summary:')
display(summary)
display(wf.best_qc)
gc.collect(); ram('After source-volume QC')


## Step 8 — Create decisive left-main QC figures


In [ ]:
figs = wf.plot_qc()
for f in figs: print('Saved:', f)
gc.collect(); ram('After figures')


## Step 9 — Package report


In [ ]:
zip_path = wf.package()
print('Report ZIP:', zip_path)
print('Drive search: https://drive.google.com/drive/u/0/search?q=OPENPLAQUE_LEFT_MAIN_LOWRAM_REPORT_BACK.zip')


After the ZIP is written, return to ChatGPT and say **Retrieve and analyze**.
